In [1]:
r"""
================================================================================
SCRIPT: FAST MANGROVE AGE RASTER IN 2025 (THREADED FOR JUPYTER / WINDOWS)
DESCRIPTION:
    - Loads valid natural tiles (mask_artif == 0) from validation shapefile.
    - Uses ThreadPoolExecutor to avoid Windows + Jupyter ProcessPoolExecutor
      BrokenProcessPool crashes while speeding up I/O and array operations.
    - For pixels present as mangrove (code 1) in 2025, finds the most recent
      non-mangrove year (code != 1) backward in time and computes age = 2025 - year.
    - Pixels continuous across 1985-2025 receive age = 41.
    - Non-mangrove pixels in 2025 receive 0 (nodata).
    - Exports single-band GeoTIFFs into D:\MANGROVES\Research_Article\Mangrove_Age_2025_Rasters.

AUTHOR: R. Walcker, CRBE
================================================================================
"""

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from pathlib import Path
import warnings
import geopandas as gpd
import numpy as np
import rasterio

os.environ.setdefault("GDAL_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

# ==============================================================================
# 0. CONFIGURATION & DIRECTORIES
# ==============================================================================
GMW_DIR = Path(r"D:\MANGROVES\GMW\V4\gmw_mng_ext_v4112_stack\reprojected_54034")
SHP_VAL_PATH = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp")
AGE_RASTER_DIR = Path(r"D:\MANGROVES\Research_Article\Mangrove_Age_2025_Rasters")
AGE_RASTER_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(1985, 2026))  # 41 bands: index 0 = 1985, index 40 = 2025
TARGET_YEAR = 2025
SKIP_EXISTING = True
MAX_WORKERS = 4  # Safe for disk I/O and RAM inside Jupyter


def process_single_tile(task: tuple[str, Path, Path]) -> str:
    """Computes and writes the 2025 mangrove age GeoTIFF for a single tile."""
    tile_id, gmw_path, out_path = task

    if SKIP_EXISTING and out_path.exists() and out_path.stat().st_size > 0:
        return f"⏭️ {tile_id}: already exists, skipped"

    try:
        with rasterio.open(gmw_path) as src:
            meta = src.meta.copy()
            stack = src.read()  # Shape: (n_bands, height, width)

        n_bands, height, width = stack.shape
        if n_bands != len(YEARS):
            return f"⚠️ {tile_id}: expected {len(YEARS)} bands, got {n_bands} (skipped)"

        age_raster = np.zeros((height, width), dtype=np.uint8)

        # Locate mangrove pixels in 2025 (last band)
        rows, cols = np.where(stack[-1] == 1)
        if len(rows) > 0:
            # History from 2024 down to 1985 for 2025 mangrove pixels: shape (40, n_pixels)
            history_rev = stack[n_bands - 2::-1, rows, cols] != 1

            # Default age if never non-mangrove in 1985..2024
            default_age = TARGET_YEAR - (YEARS[0] - 1)  # 41
            ages = np.full(len(rows), default_age, dtype=np.uint8)

            # First True along axis 0 gives the most recent non-mangrove year (0 -> 2024, 1 -> 2023, ...)
            has_non_mng = np.any(history_rev, axis=0)
            if np.any(has_non_mng):
                first_idx = np.argmax(history_rev[:, has_non_mng], axis=0)
                last_non_mng_years = np.array([YEARS[n_bands - 2 - k] for k in first_idx], dtype=np.int32)
                ages[has_non_mng] = (TARGET_YEAR - last_non_mng_years).astype(np.uint8)

            age_raster[rows, cols] = ages

        del stack

        meta.update(
            driver="GTiff",
            count=1,
            dtype=rasterio.uint8,
            nodata=0,
            compress="lzw",
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(age_raster, 1)

        return f"✅ {tile_id}: saved ({len(rows):,} px in 2025)"

    except Exception as e:
        return f"❌ {tile_id}: error -> {e}"


def main():
    print("⏳ Loading validation shapefile and filtering natural tiles...")
    gdf_val = gpd.read_file(SHP_VAL_PATH)
    tile_col_val = "clean_id" if "clean_id" in gdf_val.columns else "tile_id"
    gdf_val["tile_id_clean"] = gdf_val[tile_col_val].astype(str).str.strip()
    natural_tiles = set(gdf_val[gdf_val["mask_artif"] == 0]["tile_id_clean"])
    print(f"🎯 Found {len(natural_tiles)} natural tiles.")

    tasks = []
    for gmw_path in sorted(GMW_DIR.glob("*.tif")):
        matching_tid = next((t for t in natural_tiles if t in gmw_path.stem), None)
        if matching_tid:
            out_file = AGE_RASTER_DIR / f"mangrove_age_2025_{matching_tid}.tif"
            tasks.append((matching_tid, gmw_path, out_file))

    print(f"🚀 Processing {len(tasks)} tiles using {MAX_WORKERS} threads...")

    completed = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_single_tile, t) for t in tasks]
        for fut in as_completed(futures):
            completed += 1
            msg = fut.result()
            # Print every 50 tiles or any error/skip to keep notebook output clean
            if completed % 50 == 0 or completed == len(tasks) or msg.startswith(("❌", "⚠️")):
                print(f"[{completed}/{len(tasks)}] {msg}")

    print(f"\n🎉 All done! Outputs saved in: {AGE_RASTER_DIR}")


main()

⏳ Loading validation shapefile and filtering natural tiles...
🎯 Found 1261 natural tiles.
🚀 Processing 1261 tiles using 4 threads...
[50/1261] ✅ N01E123: saved (25,242 px in 2025)
[100/1261] ✅ N04E072: saved (0 px in 2025)
[150/1261] ✅ N06E102: saved (10,093 px in 2025)
[200/1261] ✅ N07W059: saved (19,656 px in 2025)
[250/1261] ✅ N10E118: saved (118,722 px in 2025)
[300/1261] ✅ N12W068: saved (1,515 px in 2025)
[350/1261] ✅ N15W062: saved (8,891 px in 2025)
[400/1261] ✅ N18W068: saved (5,261 px in 2025)
[450/1261] ✅ N20W156: saved (7 px in 2025)
[500/1261] ✅ N23E039: saved (5,251 px in 2025)
[550/1261] ✅ N25W098: saved (119 px in 2025)
[600/1261] ✅ N28E128: saved (50 px in 2025)
[650/1261] ✅ S01E104: saved (608,278 px in 2025)
[700/1261] ✅ S02E139: saved (37,053 px in 2025)
[750/1261] ✅ S04E116: saved (8,161 px in 2025)
[800/1261] ✅ S05W081: saved (604 px in 2025)
[850/1261] ✅ S07E140: saved (3,154 px in 2025)
[900/1261] ✅ S08E157: saved (90,880 px in 2025)
[950/1261] ✅ S11E047: saved 